In [1]:
import torch

x = torch.ones(5)  # input tensor
y = torch.zeros(3)  # expected output
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

# In this network, w and b are parameters, which we need to optimize. 
# Thus, we need to be able to compute the gradients of loss function with respect to those variables. 
# In order to do that, we set the requires_grad property of those tensors.

In [2]:
# A function that we apply to tensors to construct computational graph is in fact an object of class Function. 
# This object knows how to compute the function in the forward direction, and also how to compute its derivative during the backward propagation step. 
# A reference to the backward propagation function is stored in grad_fn property of a tensor. 

print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x000002372C560820>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x000002372B0BF9D0>


In [3]:
loss.backward()
print(w.grad)
print(b.grad)

tensor([[0.2352, 0.2938, 0.0004],
        [0.2352, 0.2938, 0.0004],
        [0.2352, 0.2938, 0.0004],
        [0.2352, 0.2938, 0.0004],
        [0.2352, 0.2938, 0.0004]])
tensor([0.2352, 0.2938, 0.0004])


In [4]:
# By default, all tensors with requires_grad=True are tracking their computational history and support gradient computation. 
# However, there are some cases when we do not need to do that, for example, when we have trained the model and just want to apply it to some input data, i.e. 
# we only want to do forward computations through the network. We can stop tracking computations by surrounding our computation code with torch.no_grad() block:
z = torch.matmul(x, w)+b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)

True
False


In a forward pass, autograd does two things simultaneously:

- run the requested operation to compute a resulting tensor
- maintain the operation’s _gradient function_ in the DAG.

The backward pass kicks off when `.backward()` is called on the DAG root. `autograd` then:

- computes the gradients from each `.grad_fn`,
- accumulates them in the respective tensor’s `.grad` attribute
- using the chain rule, propagates all the way to the leaf tensors.

<div class="alert alert-block alert-info">
<b>NOTE:</b>
<P>DAGs are dynamic in PyTorch An important thing to note is that the graph is recreated from scratch; after each .backward() call, autograd starts populating a new graph. This is exactly what allows you to use control flow statements in your model; you can change the shape, size and operations at every iteration if needed.</P>

<p>The new DAG is created during the next forward pass, as operations execute</p>
</div>
